In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

In [8]:
TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'

In [9]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [11]:
print(f"Train: {train.shape}  |  Days in train: {sorted(train['day'].unique())}")
print(f"Test : {test.shape}   |  Days in test : {sorted(test['day'].unique())}")

Train: (77299, 11)  |  Days in train: [np.int64(48), np.int64(49)]
Test : (41778, 10)   |  Days in test : [np.int64(49)]


In [12]:
def parse_time(df: pd.DataFrame) -> pd.DataFrame:
    parts          = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']     = parts[0]
    df['minute']   = parts[1]
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15   # 0 … 95
    return df

In [13]:
train = parse_time(train)
test  = parse_time(test)

In [14]:
demand_lookup = (
    train
    .set_index(['geohash', 'day', 'time_slot'])['demand']
)

In [39]:
def add_lags(df: pd.DataFrame) -> pd.DataFrame:
    # --- 1-DAY LAG (Same time yesterday) ---
    idx_1day = pd.MultiIndex.from_arrays([
        df['geohash'], df['day'] - 1, df['time_slot']
    ])
    df['demand_lag_1day'] = demand_lookup.reindex(idx_1day).values
    
    # --- 15-MINUTE LAG (1 time slot ago) ---
    slot_15m = df['time_slot'] - 1
    day_15m = df['day']
    
    # Handle midnight rollover (if slot goes below 0, go to day-1, slot 95)
    rollover_15m = slot_15m < 0
    day_15m = np.where(rollover_15m, day_15m - 1, day_15m)
    slot_15m = np.where(rollover_15m, slot_15m + 96, slot_15m)
    
    idx_15m = pd.MultiIndex.from_arrays([df['geohash'], day_15m, slot_15m])
    df['demand_lag_15min'] = demand_lookup.reindex(idx_15m).values

    # --- 1-HOUR LAG (4 time slots ago) ---
    slot_1hr = df['time_slot'] - 4
    day_1hr = df['day']
    
    # Handle midnight rollover 
    rollover_1hr = slot_1hr < 0
    day_1hr = np.where(rollover_1hr, day_1hr - 1, day_1hr)
    slot_1hr = np.where(rollover_1hr, slot_1hr + 96, slot_1hr)
    
    idx_1hr = pd.MultiIndex.from_arrays([df['geohash'], day_1hr, slot_1hr])
    df['demand_lag_1hr'] = demand_lookup.reindex(idx_1hr).values

    return df

In [40]:
train = add_lags(train)
test  = add_lags(test)

In [17]:
ref = train[train['day'] == 48]

In [18]:
geo_stats = (
    ref.groupby('geohash')['demand']
    .agg(geo_mean='mean', geo_std='std', geo_median='median')
)

In [19]:
geo_slot_mean = (
    ref.groupby(['geohash', 'time_slot'])['demand']
    .mean()
    .rename('geo_slot_mean')
)

In [20]:
def add_geo_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(geo_stats.reset_index(),     on='geohash',                 how='left')
    df = df.merge(geo_slot_mean.reset_index(), on=['geohash', 'time_slot'],  how='left')
    return df

In [21]:
train = add_geo_features(train)
test  = add_geo_features(test)

In [22]:
global_mean = train['demand'].mean()

In [23]:
for df in [train, test]:
    df['Temperature'] = df['Temperature'].fillna(-999)

In [24]:
train['demand_lag_1day'] = train['demand_lag_1day'].fillna(train['geo_mean'])
test['demand_lag_1day']  = test['demand_lag_1day'].fillna(test['geo_mean'])

In [25]:
for df in [train, test]:
    df['geo_std']      = df['geo_std'].fillna(0)
    df['geo_mean']     = df['geo_mean'].fillna(global_mean)
    df['geo_median']   = df['geo_median'].fillna(global_mean)
    df['geo_slot_mean']= df['geo_slot_mean'].fillna(global_mean)
    df['demand_lag_1day'] = df['demand_lag_1day'].fillna(global_mean)

In [26]:
CAT_FEATURES = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in CAT_FEATURES:
    for df in [train, test]:
        df[col] = df[col].fillna('Unknown').astype(str)

In [27]:
FEATURES = [
    # Spatial
    'geohash',
    # Temporal
    'day', 'hour', 'minute', 'time_slot',
    # Road properties
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    # Weather
    'Temperature', 'Weather',
    # Lag
    'demand_lag_1day',
    # Geohash aggregates
    'geo_mean', 'geo_std', 'geo_median', 'geo_slot_mean',
]
 
TARGET = 'demand'

In [28]:
train_df = train[train['day'] == 48].copy()
val_df   = train[train['day'] == 49].copy()
 
print(f"\nSplit → train: {len(train_df)} rows | val: {len(val_df)} rows")
 
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val,   y_val   = val_df[FEATURES],   val_df[TARGET]


Split → train: 69427 rows | val: 7872 rows


In [29]:
MODEL_PARAMS = dict(
    iterations   = 3000,
    learning_rate= 0.05,
    depth        = 6,
    loss_function= 'RMSE',
    eval_metric  = 'RMSE',
    random_seed  = 42,
    od_type      = 'Iter',   # early stop: stop if no improvement for od_wait iters
    od_wait      = 150,
    verbose      = 300,
)

In [30]:
model = CatBoostRegressor(**MODEL_PARAMS)
model.fit(
    X_train, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_val, y_val),
)

0:	learn: 0.1350801	test: 0.1421459	best: 0.1421459 (0)	total: 161ms	remaining: 8m 2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.09904196366
bestIteration = 77

Shrink model to first 78 iterations.


CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=3000, learning_rate=0.05, loss_function='RMSE', od_type='Iter', od_wait=150, random_seed=42, verbose=300)

In [32]:
from sklearn.metrics import r2_score
val_preds = np.clip(model.predict(X_val), 0, None)   # demand ≥ 0
r2        = r2_score(y_val, val_preds)
score     = max(0.0, 100.0 * r2)
 
print(f"\n{'─'*45}")
print(f"  Validation R²             : {r2:.5f}")
print(f"  Estimated competition score: {score:.2f} / 100")
print(f"{'─'*45}")


─────────────────────────────────────────────
  Validation R²             : 0.53204
  Estimated competition score: 53.20 / 100
─────────────────────────────────────────────


In [33]:
fi = pd.Series(model.get_feature_importance(), index=FEATURES).sort_values(ascending=False)
print("\nTop 10 feature importances:")
print(fi.head(10).to_string())


Top 10 feature importances:
geo_slot_mean      93.623919
RoadType            5.580238
geo_median          0.330796
geo_std             0.184192
geo_mean            0.084652
NumberofLanes       0.066053
LargeVehicles       0.061087
demand_lag_1day     0.038052
Weather             0.017229
Landmarks           0.009357


In [34]:
print("\nRetraining on full training data for final submission …")


Retraining on full training data for final submission …


In [35]:
best_iters = model.best_iteration_   # use the iteration count found via early stopping
final_model = CatBoostRegressor(**{**MODEL_PARAMS, 'iterations': best_iters, 'od_type': 'IncToDec', 'od_wait': 10, 'verbose': 200})
final_model.fit(
    train[FEATURES], train[TARGET],
    cat_features=CAT_FEATURES,
)

0:	learn: 0.1356532	total: 50ms	remaining: 3.8s
76:	learn: 0.0189284	total: 3.76s	remaining: 0us


CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=77, learning_rate=0.05, loss_function='RMSE', od_type='IncToDec', od_wait=10, random_seed=42, verbose=200)

In [36]:
test_preds = np.clip(final_model.predict(test[FEATURES]), 0, None)

In [37]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
OUT_PATH   = 'submission.csv'
submission.to_csv(OUT_PATH, index=False)

In [38]:
print(f"\nSubmission saved → {OUT_PATH}")
print(f"Shape : {submission.shape}  (expected 41778 × 2)")
print(submission.head())


Submission saved → submission.csv
Shape : (41778, 2)  (expected 41778 × 2)
   Index    demand
0      0  0.069625
1      1  0.056758
2      2  0.041482
3      3  0.069288
4      4  0.102201
